In [3]:
import pandas as pd
# --- 1. Configuration des fichiers ---
fichier_entree = "gbm_full_features_aaindex1.csv" 
fichier_sortie = "all_sequences.fasta" 

# --- 2. Lecture du fichier CSV ---
df = pd.read_csv(fichier_entree, sep=',')

# --- COMPTAGE 1 : Nombre total ---
# len(df) compte le nombre total de lignes dans le tableau
nombre_total = len(df)
print(f"Nombre total de lignes (séquences) dans le CSV : {nombre_total}")

# --- 3. Suppression des doublons ---
df_unique = df.drop_duplicates(subset=['Gene'], keep='first')
# --- 1. Identifier les séquences trop longues (> 5000) ---
# On crée un masque (filtre) basé sur la longueur de la chaîne de caractères
trop_longues = df_unique[df_unique['Sequence'].str.len() > 5000]

print(f"\n--- SÉQUENCES TROP LONGUES (> 5000) : {len(trop_longues)} trouvées ---")
if len(trop_longues) > 0:
    for index, row in trop_longues.iterrows():
        longueur = len(row['Sequence'])
        print(f"Gène : {row['Gene']} | ID : {row['UniProt ID']} | Longueur : {longueur}")

# --- 2. Identifier les séquences trop courtes (< 10) ---
trop_courtes = df_unique[df_unique['Sequence'].str.len() < 10]

print(f"\n--- SÉQUENCES TROP COURTES (< 10) : {len(trop_courtes)} trouvées ---")
if len(trop_courtes) > 0:
    for index, row in trop_courtes.iterrows():
        longueur = len(row['Sequence'])
        print(f"Gène : {row['Gene']} | ID : {row['UniProt ID']} | Longueur : {longueur}")

# --- 3. Nettoyer le tableau pour le FASTA final ---
# On garde uniquement les séquences qui sont >= 10 ET <= 5000
df_valide = df_unique[(df_unique['Sequence'].str.len() >= 10) & 
                      (df_unique['Sequence'].str.len() <= 5000)]

print(f"\nNombre de séquences valides à exporter : {len(df_valide)}")

# --- 4. Création du fichier FASTA avec les séquences valides ---
fichier_sortie = "all_sequences_valides.fasta" 
with open(fichier_sortie, "w") as fasta_file:
    for index, row in df_valide.iterrows():
        en_tete = f">{row['UniProt ID']}_{row['Gene']}\n"
        fasta_file.write(en_tete)
        fasta_file.write(f"{row['Sequence']}\n")

print(f"Fichier '{fichier_sortie}' créé avec succès et prêt à être soumis !")

Nombre total de lignes (séquences) dans le CSV : 18114

--- SÉQUENCES TROP LONGUES (> 5000) : 24 trouvées ---
Gène : FSIP2 | ID : Q5CZC0 | Longueur : 6907
Gène : UBR4 | ID : Q5T4S7 | Longueur : 5183
Gène : MUC5A | ID : P98088 | Longueur : 5654
Gène : AHNK | ID : Q09666 | Longueur : 5890
Gène : AHNK2 | ID : Q8IVF2 | Longueur : 5795
Gène : DYST | ID : Q03001 | Longueur : 7570
Gène : AGRV1 | ID : Q8WXG9 | Longueur : 6306
Gène : HMCN1 | ID : Q96RW7 | Longueur : 5635
Gène : HYDIN | ID : Q4G0P3 | Longueur : 5121
Gène : K1109 | ID : Q2LD37 | Longueur : 5005
Gène : KMT2D | ID : O14686 | Longueur : 5537
Gène : MACF1 | ID : Q9UPN3 | Longueur : 7388
Gène : MUC12 | ID : Q9UKN1 | Longueur : 5478
Gène : MUC16 | ID : Q8WXI7 | Longueur : 14507
Gène : MUC4 | ID : Q99102 | Longueur : 5412
Gène : MUC5B | ID : Q9HC84 | Longueur : 5762
Gène : NEBU | ID : P20929 | Longueur : 8525
Gène : OBSCN | ID : Q5VST9 | Longueur : 7968
Gène : PCLO | ID : Q9Y6V0 | Longueur : 5142
Gène : RN213 | ID : Q63HN8 | Longueur : 

In [4]:
# --- CELLULE DE DÉTECTION DES MAUVAIS CARACTÈRES ---

# 1. Définition des 20 acides aminés acceptés
acides_amines_standards = set("ACDEFGHIKLMNPQRSTVWY")

def trouver_intrus(sequence):
    seq = str(sequence).upper().strip() 
    intrus = set(seq) - acides_amines_standards
    return "".join(intrus)

# 2. Application de la recherche (on suppose que votre tableau s'appelle df_valide)
df_valide['Intrus'] = df_valide['Sequence'].apply(trouver_intrus)

# 3. Isolation des séquences problématiques
df_erreurs = df_valide[df_valide['Intrus'] != ""]

print(f"Séquences avec caractères invalides : {len(df_erreurs)} trouvées.\n")

# Dans Jupyter, on utilise display() pour avoir un bel affichage du tableau
if len(df_erreurs) > 0:
    print("Voici un aperçu des gènes rejetés et de leurs mauvais caractères :")
    display(df_erreurs[['UniProt ID', 'Gene', 'Intrus']].head(20)) # Affiche les 20 premiers

# 4. Création du tableau final propre (sans les intrus)
df_final = df_valide[df_valide['Intrus'] == ""]
print(f"\nNombre de séquences parfaites restantes : {len(df_final)}")

Séquences avec caractères invalides : 3 trouvées.

Voici un aperçu des gènes rejetés et de leurs mauvais caractères :


C:\Users\nicol\AppData\Local\Temp\ipykernel_26036\2160586114.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valide['Intrus'] = df_valide['Sequence'].apply(trouver_intrus)


,UniProt ID,Gene,Intrus
1260,P36969,GPX4,U
1443,P55073,IOD3,U
17466,Q9NNW7,TRXR2,U



Nombre de séquences parfaites restantes : 4220


In [5]:
# --- CELLULE DE CORRECTION (REMPLACEMENT U -> C) ---

# 1. On remplace les 'U' par des 'C' dans notre tableau principal
df_valide['Sequence'] = df_valide['Sequence'].str.replace('U', 'C', regex=False)
print("Correction terminée : les 'U' ont bien été remplacés par des 'C'.")

# 2. On met à jour la colonne 'Intrus' pour vérifier que tout est propre
df_valide['Intrus'] = df_valide['Sequence'].apply(trouver_intrus)

# 3. On recrée df_final (qui va maintenant inclure vos 3 séquences récupérées)
df_final = df_valide[df_valide['Intrus'] == ""]

print(f"Nouveau nombre total de séquences prêtes pour l'export : {len(df_final)}")

Correction terminée : les 'U' ont bien été remplacés par des 'C'.
Nouveau nombre total de séquences prêtes pour l'export : 4223


C:\Users\nicol\AppData\Local\Temp\ipykernel_26036\3676966495.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valide['Sequence'] = df_valide['Sequence'].str.replace('U', 'C', regex=False)
C:\Users\nicol\AppData\Local\Temp\ipykernel_26036\3676966495.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_valide['Intrus'] = df_valide['Sequence'].apply(trouver_intrus)


In [6]:
# --- CELLULE D'EXPORT DU FASTA FINAL ---

fichier_sortie = "sequences_netsurfp_final.fasta" 

# On ouvre un nouveau fichier en mode écriture ("w")
with open(fichier_sortie, "w") as fasta_file:
    # On parcourt chaque ligne de notre tableau propre
    for index, row in df_final.iterrows():
        # Ultime vérification : on s'assure que la séquence est en majuscules et sans espaces
        seq_propre = str(row['Sequence']).upper().strip()
        
        # Création de la ligne d'en-tête (ex: >P12345_GENE)
        en_tete = f">{row['UniProt ID']}_{row['Gene']}\n"
        
        # On écrit l'en-tête puis la séquence en dessous
        fasta_file.write(en_tete)
        fasta_file.write(f"{seq_propre}\n")

print(f"Succès ! Le fichier '{fichier_sortie}' a été généré.")
print(f"Il contient {len(df_final)} séquences parfaitement formatées et prêtes pour NetSurfP !")

Succès ! Le fichier 'sequences_netsurfp_final.fasta' a été généré.
Il contient 4223 séquences parfaitement formatées et prêtes pour NetSurfP !
